In [1]:
# ==========================================
# FACE RECOGNITION - TRAINING (FINAL)
# ==========================================

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from collections import defaultdict
import os

# ======================
# DEVICE
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# DATA PATH (FIXED)
# ======================
lfw_path = r"Z:\lfw\lfw-deepfunneled\lfw-deepfunneled"

print("Dataset path:", lfw_path)
print("Path exists:", os.path.exists(lfw_path))
print("Sample folders:", os.listdir(lfw_path)[:5])

# ======================
# TRANSFORM
# ======================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ======================
# LOAD DATASET
# ======================
dataset = datasets.ImageFolder(root=lfw_path, transform=transform)

print("Total classes:", len(dataset.classes))

# ======================
# REDUCE DATASET SIZE
# ======================
class_count = defaultdict(int)
selected_indices = []

max_images_per_class = 20

for idx, (img, label) in enumerate(dataset):
    if class_count[label] < max_images_per_class:
        selected_indices.append(idx)
        class_count[label] += 1

dataset = torch.utils.data.Subset(dataset, selected_indices)

print("Reduced dataset size:", len(dataset))

# ======================
# DATALOADER
# ======================
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# ======================
# MODEL
# ======================
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_classes = len(dataset.dataset.classes)

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ======================
# TRAINING
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print("Training started...")

for epoch in range(2):
    total_loss = 0
    model.train()
    
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        
        out = model(x)
        loss = criterion(out, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if i % 20 == 0:
            print(f"Batch {i} running...")
    
    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

print("Training completed!")

# ======================
# SAVE MODEL + CLASSES
# ======================
torch.save({
    "model_state": model.state_dict(),
    "classes": dataset.dataset.classes
}, "lfw_face_model.pth")

print("Model saved successfully!")

Dataset path: Z:\lfw\lfw-deepfunneled\lfw-deepfunneled
Path exists: True
Sample folders: ['Aaron_Eckhart', 'Aaron_Guiel', 'Aaron_Patterson', 'Aaron_Peirsol', 'Aaron_Pena']
Total classes: 5749
Reduced dataset size: 11450
Training started...
Batch 0 running...
Batch 20 running...
Batch 40 running...
Batch 60 running...
Batch 80 running...
Batch 100 running...
Batch 120 running...
Batch 140 running...
Batch 160 running...
Batch 180 running...
Batch 200 running...
Batch 220 running...
Batch 240 running...
Batch 260 running...
Batch 280 running...
Batch 300 running...
Batch 320 running...
Batch 340 running...
Batch 360 running...
Batch 380 running...
Batch 400 running...
Batch 420 running...
Batch 440 running...
Batch 460 running...
Batch 480 running...
Batch 500 running...
Batch 520 running...
Batch 540 running...
Batch 560 running...
Batch 580 running...
Batch 600 running...
Batch 620 running...
Batch 640 running...
Batch 660 running...
Batch 680 running...
Batch 700 running...
Epoch 1 Lo

In [ ]:
# ==========================================
# FACE RECOGNITION - PREDICTION
# ==========================================

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt

# ======================
# DEVICE
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# TRANSFORM
# ======================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ======================
# LOAD MODEL
# ======================
checkpoint = torch.load("lfw_face_model.pth")

classes = checkpoint["classes"]

model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(checkpoint["model_state"])

model = model.to(device)
model.eval()

print("Model loaded!")

# ======================
# TEST IMAGE
# ======================
img_path = r"Z:\images.jpg"  # 🔥 change this

img = Image.open(img_path).convert("RGB")

inp = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(inp)
    pred = torch.argmax(output).item()
    confidence = torch.softmax(output, dim=1)[0][pred].item()

label = classes[pred]

print(f"Prediction: {label}")
print(f"Confidence: {confidence:.2f}")

plt.imshow(img)
plt.title(f"{label} ({confidence:.2f})")
plt.axis("off")
plt.show()